In [0]:
%run ../functions/functions

In [0]:
silver_path_s = f"abfss://silver@stgbbb.dfs.core.windows.net/balancacomercial/EXP_MUN_CONSOLIDADA/"
silver_path_ncm_sh = f"abfss://silver@stgbbb.dfs.core.windows.net/balancacomercial/NCM_SH_CONSOLIDADA/" 
silver_path_ncm = f"abfss://silver@stgbbb.dfs.core.windows.net/balancacomercial/NCM/"

In [0]:
df = spark.read.format("delta").load(silver_path_s)
df_sh = spark.read.format("delta").load(silver_path_ncm_sh)
df_ncm = spark.read.format("delta").load(silver_path_ncm)

In [0]:
df = df.withColumn("sk_comex_regioes", F.concat(F.col("SG_UF_MUN"), F.col("CO_MUN")))

cols = ["sk_comex_regioes"] + [c for c in df.columns if c != "sk_comex_regioes"]

df = df.select(cols)

df_organizado = df.drop("SK_EXPORTACAO_MUN")

In [0]:
df_organizado.createOrReplaceTempView("df_exp_mun")
df_sh.createOrReplaceTempView("df_sh")
df_ncm.createOrReplaceTempView("df_ncm")

In [0]:
%sql
select count(*) from df_exp_mun

In [0]:
df_organizado.select("*").count()

In [0]:
%sql

  select
    c.CO_NCM as sk_comex_ncm,
    e.sk_comex_regioes,
    e.CO_PAIS,
    e.SG_UF_MUN,
    e.CO_MUN,
    e.CO_ANO,
    e.CO_MES,
    e.VL_FOB,
    e.SH4
    -- m.CO_SH6
  from df_exp_mun as e
  join df_sh as m on e.SH4 = m.CO_SH4
  join df_ncm as c on m.CO_SH6 = c.CO_SH6

In [0]:
df_organizado.printSchema()